# 02 — Build features

Reads claims + premiums CSVs, builds a leakage-safe training table
`{catalog}.{schema}.claim_severity_features` (one row per closed claim).

Runs on the Shared all-purpose cluster.

### Step 1 — Configure outputs

Read widgets for catalog, schema, landing path, fixture `source_path`, and `project_src` (where the shared Python package lives). The target Delta table is `{catalog}.{schema}.claim_severity_features`.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "ml")
dbutils.widgets.text("landing_path", "/Volumes/actuarial/ml/landing")
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("project_src", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
source_path = dbutils.widgets.get("source_path").rstrip("/")
project_src = dbutils.widgets.get("project_src").rstrip("/")

table_name = f"{catalog}.{schema}.claim_severity_features"
print(f"Writing {table_name}")

### Step 2 — Load CSVs and build features

Put `project_src` on `sys.path`, then load claims and premiums CSVs (prefer the UC Volume; fall back to workspace fixtures). Call `build_claim_severity_features` to produce one leakage-safe row per closed claim: first-report features plus the ultimate incurred label.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

candidates = [Path(project_src)]
if project_src and not project_src.startswith("/Workspace"):
    candidates.append(Path("/Workspace") / project_src.lstrip("/"))
for p in candidates:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    raise FileNotFoundError(f"project_src not found: {project_src}")

from ml_pipeline_demo.features import build_claim_severity_features


def _resolve_csv(volume_path: str, filename: str, source_subdir_file: str) -> str:
    """Prefer Volume path; fall back to workspace fixtures if volume read is empty."""
    vol = f"{landing_path}/{volume_path}/{filename}"
    try:
        n = spark.read.csv(vol, header=True, inferSchema=False).count()
        if n > 0:
            print(f"Using volume CSV {vol} rows={n}")
            return vol
    except Exception as exc:  # noqa: BLE001
        print(f"Volume read skipped for {vol}: {exc}")

    src_root = Path(source_path)
    if not src_root.exists():
        alt = Path("/Workspace") / source_path.lstrip("/")
        if alt.exists():
            src_root = alt
    local = src_root / source_subdir_file
    if not local.exists():
        raise FileNotFoundError(f"Neither volume nor source CSV found: {vol} / {local}")
    print(f"Using workspace CSV {local}")
    return str(local)


# Prefer Volume CSVs; fall back to workspace fixtures if the volume is empty.
claims_uri = _resolve_csv("claims", "claims_bordereau.csv", "claims_bordereau.csv")
premiums_uri = _resolve_csv("premiums", "premium_bordereau.csv", "premium_bordereau.csv")

claims_pdf = spark.read.csv(claims_uri, header=True, inferSchema=False).toPandas()
policies_pdf = spark.read.csv(premiums_uri, header=True, inferSchema=False).toPandas()
print(f"claims rows={len(claims_pdf)} policies rows={len(policies_pdf)}")

# Closed claims only; first-report features + ultimate_incurred label.
features_pdf = build_claim_severity_features(claims_pdf, policies_pdf)
print(f"feature rows={len(features_pdf)}")
display(features_pdf.head(10))

### Step 3 — Write the Delta feature table

Normalize pandas dtypes so Spark/Arrow conversion is reliable, overwrite the Delta table (including schema), and assert we wrote roughly 1100+ closed-claim rows before continuing the pipeline.

In [ ]:
# Ensure Arrow-friendly dtypes before Spark conversion.
features_pdf = features_pdf.copy()
features_pdf["claim_id"] = features_pdf["claim_id"].astype(str)
features_pdf["policy_id"] = features_pdf["policy_id"].astype(str)
for col in ["peril_type", "wind_risk_band", "building_type", "mitigation_flag", "region_name"]:
    features_pdf[col] = features_pdf[col].astype(str)
for col in ["first_incurred", "report_lag_days", "sum_insured", "ultimate_incurred"]:
    features_pdf[col] = pd.to_numeric(features_pdf[col], errors="coerce")

features_df = spark.createDataFrame(features_pdf)
(
    features_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

count = spark.table(table_name).count()
print(f"Wrote {count} rows to {table_name} (pandas feature rows={len(features_pdf)})")
assert count > 1000, f"Expected ~1100+ closed claims, got {count}"
print("Feature build complete.")